# 18 - Inverse Reinforcement Learning

## Learning Objectives
1. Understand the core IRL problem: recover reward from expert demonstrations
2. Implement Behavioral Cloning and show its distributional shift failure
3. Build MaxEnt IRL on GridWorld and visualize recovered reward function
4. Compare BC vs MaxEnt IRL vs GAIL-approximation on data efficiency


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Packages loaded: numpy', np.__version__)
print('IRL demo: GridWorld Behavioral Cloning + MaxEnt IRL + GAIL approx')


## Level 1: Behavioral Cloning on GridWorld

BC = supervised learning of pi(a|s) from (s, a) expert demonstrations.
Predict the expert's action at each state using logistic regression.
Track: training accuracy vs test accuracy vs on-policy performance (compounding errors).


In [ ]:
# GridWorld environment
class GridWorld:
    def __init__(self, rows=5, cols=5):
        self.rows = rows
        self.cols = cols
        self.goal = (rows-1, cols-1)
        self.n_states = rows * cols
        self.n_actions = 4  # up, down, left, right
        self.deltas = [(-1,0),(1,0),(0,-1),(0,1)]

    def reset(self, pos=None):
        self.pos = pos if pos else (0, 0)
        return self._state()

    def _state(self):
        return self.pos[0] * self.cols + self.pos[1]

    def step(self, action):
        r, c = self.pos
        dr, dc = self.deltas[action]
        nr = max(0, min(self.rows-1, r+dr))
        nc = max(0, min(self.cols-1, c+dc))
        self.pos = (nr, nc)
        done = self.pos == self.goal
        reward = 1.0 if done else -0.01
        return self._state(), reward, done

    def optimal_action(self, s):
        # Optimal policy: move toward goal (greedy by Manhattan distance)
        r, c = s // self.cols, s % self.cols
        gr, gc = self.goal
        candidates = []
        for a, (dr, dc) in enumerate(self.deltas):
            nr = max(0, min(self.rows-1, r+dr))
            nc = max(0, min(self.cols-1, c+dc))
            dist = abs(nr - gr) + abs(nc - gc)
            candidates.append((dist, a))
        return min(candidates)[1]


def generate_expert_demos(n_trajectories=20, noise=0.1):
    env = GridWorld()
    demos = []
    for _ in range(n_trajectories):
        s = env.reset()
        traj = []
        done = False
        steps = 0
        while not done and steps < 50:
            if np.random.random() < noise:
                a = np.random.randint(env.n_actions)  # noisy expert
            else:
                a = env.optimal_action(s)  # optimal action
            traj.append((s, a))
            s, _, done = env.step(a)
            steps += 1
        demos.append(traj)
    return demos


def train_bc(demos):
    # Extract (s, a) pairs for supervised learning
    sa_pairs = [(s, a) for traj in demos for s, a in traj]
    X = np.array([s for s, a in sa_pairs])
    y = np.array([a for s, a in sa_pairs])

    # One-hot encode state
    X_ohe = np.eye(25)[X]

    clf = LogisticRegression(C=1.0, max_iter=500, random_state=42)
    clf.fit(X_ohe, y)
    train_acc = clf.score(X_ohe, y)
    return clf, train_acc


def evaluate_bc_policy(clf, n_episodes=100, max_steps=50):
    env = GridWorld()
    success_count = 0
    total_steps = []
    for _ in range(n_episodes):
        s = env.reset()
        done = False
        steps = 0
        while not done and steps < max_steps:
            s_ohe = np.eye(25)[s].reshape(1, -1)
            a = clf.predict(s_ohe)[0]
            s, _, done = env.step(a)
            steps += 1
        if done:
            success_count += 1
            total_steps.append(steps)
    return success_count / n_episodes, (np.mean(total_steps) if total_steps else max_steps)


# Train BC with different numbers of demonstrations
demo_counts = [1, 5, 10, 20, 50]
bc_success = []
for n_demos in demo_counts:
    demos = generate_expert_demos(n_trajectories=n_demos, noise=0.05)
    clf_bc, train_acc = train_bc(demos)
    succ, steps = evaluate_bc_policy(clf_bc)
    bc_success.append(succ)
    print(f'BC with {n_demos:2d} demos: train_acc={train_acc:.2f}, success_rate={succ:.2f}')


## Level 2: MaxEnt IRL on GridWorld

MaxEnt IRL: find reward R that maximizes expert feature count matching.
Gradient = E_expert[features] - E_policy[features]
Iterate: update reward weights -> resolve policy -> recompute policy features.
Visualize the recovered reward function as a heatmap.


In [ ]:
def softmax_value_iteration(reward_weights, features, gamma=0.9, n_iter=20):
    # features: (n_states, n_features), reward_weights: (n_features,)
    # Returns state-action values Q(s,a) and stochastic policy
    n_states = 25
    n_actions = 4
    rows, cols = 5, 5
    deltas = [(-1,0),(1,0),(0,-1),(0,1)]

    # Build reward per state from features and weights
    R = features @ reward_weights  # shape: (n_states,)

    # Build transition matrix
    def next_state(s, a):
        r, c = s // cols, s % cols
        dr, dc = deltas[a]
        nr = max(0, min(rows-1, r+dr))
        nc = max(0, min(cols-1, c+dc))
        return nr * cols + nc

    # Compute Q via value iteration
    V = np.zeros(n_states)
    for _ in range(n_iter):
        Q = np.zeros((n_states, n_actions))
        for s in range(n_states):
            for a in range(n_actions):
                s2 = next_state(s, a)
                Q[s, a] = R[s] + gamma * V[s2]
        V = np.max(Q, axis=1)

    # Softmax policy (MaxEnt distribution over trajectories)
    Q_shifted = Q - Q.max(axis=1, keepdims=True)
    exp_Q = np.exp(Q_shifted)
    policy = exp_Q / exp_Q.sum(axis=1, keepdims=True)
    return policy, Q


def compute_expected_feature_counts(policy, features, n_trajectories=500, max_len=30):
    n_states, n_features = features.shape
    rows, cols = 5, 5
    deltas = [(-1,0),(1,0),(0,-1),(0,1)]

    def next_state(s, a):
        r, c = s // cols, s % cols
        dr, dc = deltas[a]
        nr = max(0, min(rows-1, r+dr))
        nc = max(0, min(cols-1, c+dc))
        return nr * cols + nc

    feature_counts = np.zeros(n_features)
    for _ in range(n_trajectories):
        s = 0  # always start from (0,0)
        for _ in range(max_len):
            feature_counts += features[s]
            a = np.random.choice(4, p=policy[s])
            s = next_state(s, a)
            if s == 24:  # goal state
                feature_counts += features[s]
                break
    return feature_counts / n_trajectories


def greedy_value_iteration(reward_weights, features, gamma=0.9, n_iter=80):
    """Standard (greedy) value iteration. Returns Q-table and greedy policy."""
    rows, cols = 5, 5
    n_states, n_features = features.shape
    n_actions = 4
    deltas = [(-1,0),(1,0),(0,-1),(0,1)]

    def next_state(s, a):
        r, c = s // cols, s % cols
        dr, dc = deltas[a]
        return max(0, min(rows-1, r+dr)) * cols + max(0, min(cols-1, c+dc))

    R = features @ reward_weights
    V = np.zeros(n_states)
    for _ in range(n_iter):
        Q = np.zeros((n_states, n_actions))
        for s in range(n_states):
            for a in range(n_actions):
                Q[s, a] = R[s] + gamma * V[next_state(s, a)]
        V = np.max(Q, axis=1)
    return Q  # evaluate with np.argmax(Q, axis=1)


def run_maxent_irl(demos, n_features=25, n_iters=60, lr=0.05):
    """MaxEnt IRL: infer reward weights from expert demonstrations.
    Uses standard value iteration (greedy policy) for inner RL problem.
    Gradient: expert_feature_counts - policy_feature_counts.
    """
    features = np.eye(n_features)  # one-hot state features

    # Expert feature counts (avg trajectory state visitation)
    expert_counts = np.zeros(n_features)
    for traj in demos:
        for s, a in traj:
            expert_counts += features[s]
    expert_counts /= len(demos)

    reward_weights = np.zeros(n_features)
    loss_history = []

    # Helper: compute feature counts under greedy policy
    rows, cols = 5, 5
    n_actions = 4
    deltas = [(-1,0),(1,0),(0,-1),(0,1)]

    def ns(s, a):
        r, c = s // cols, s % cols
        dr, dc = deltas[a]
        return max(0, min(rows-1, r+dr)) * cols + max(0, min(cols-1, c+dc))

    for iteration in range(n_iters):
        # Solve inner RL with greedy value iteration
        Q = greedy_value_iteration(reward_weights, features)
        greedy_pol = np.argmax(Q, axis=1)  # deterministic greedy policy

        # Compute greedy policy feature counts
        policy_counts = np.zeros(n_features)
        for _ in range(300):  # 300 trajectories for stable gradient
            s = 0  # start at (0,0)
            for _ in range(40):
                policy_counts += features[s]
                a = greedy_pol[s]
                s = ns(s, a)
                if s == 24:
                    policy_counts += features[s]
                    break
        policy_counts /= 300

        # Gradient ascent on reward weights
        grad = expert_counts - policy_counts
        reward_weights += lr * grad
        loss_history.append(np.sum((expert_counts - policy_counts) ** 2))

    return reward_weights, loss_history


# Generate expert demonstrations
np.random.seed(42)
expert_demos = generate_expert_demos(n_trajectories=20, noise=0.05)
print(f'Expert demos: {len(expert_demos)} trajectories, '
      f'{sum(len(t) for t in expert_demos)} total steps')

print('Running MaxEnt IRL (60 iterations)...')
reward_weights, irl_losses = run_maxent_irl(expert_demos)
print(f'IRL final feature-matching loss: {irl_losses[-1]:.4f}')

# Visualize recovered reward heatmap
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# True reward (1 at goal, 0 elsewhere)
true_R = np.zeros(25)
true_R[24] = 1.0
axes[0].imshow(true_R.reshape(5, 5), cmap='RdYlGn', vmin=-0.5, vmax=1.0)
axes[0].set_title('True Reward', fontsize=12)
axes[0].set_facecolor('#f0f0f0')
for i in range(5):
    for j in range(5):
        axes[0].text(j, i, f'{true_R[i*5+j]:.1f}',
                     ha='center', va='center', fontsize=9)

# Recovered reward
im = axes[1].imshow(reward_weights.reshape(5, 5), cmap='RdYlGn')
axes[1].set_title('MaxEnt IRL: Recovered Reward', fontsize=12)
plt.colorbar(im, ax=axes[1])
for i in range(5):
    for j in range(5):
        axes[1].text(j, i, f'{reward_weights[i*5+j]:.2f}',
                     ha='center', va='center', fontsize=7)

# IRL convergence
axes[2].plot(irl_losses, color='#2c7bb6', linewidth=2)
axes[2].set_xlabel('IRL Iteration', fontsize=12)
axes[2].set_ylabel('Feature Matching Loss (MSE)', fontsize=12)
axes[2].set_title('MaxEnt IRL Convergence', fontsize=12)
axes[2].set_facecolor('#f8f8f8')
axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('/tmp/irl_reward_recovery.png', dpi=100, bbox_inches='tight')
plt.show()

# Evaluate policy trained on recovered reward (greedy argmax Q)
features_eval = np.eye(25)
Q_recovered = greedy_value_iteration(reward_weights, features_eval)
env_eval = GridWorld()
successes = 0
for _ in range(200):
    s = env_eval.reset()
    done = False
    for _ in range(50):
        a = np.argmax(Q_recovered[s])
        s, _, done = env_eval.step(a)
        if done:
            successes += 1
            break
print(f'Policy trained on recovered reward: {successes/200:.2f} success rate')


## Real-World Example 1: Distributional Shift in BC

Show how BC error compounds over long horizons.
Compare performance on short (T=10) vs long (T=50) evaluation sequences.
Quantify the O(T^2) compounding error phenomenon.


In [ ]:
def evaluate_bc_over_horizon(clf, start_states, horizon_lengths):
    env = GridWorld()
    results = {}
    for H in horizon_lengths:
        rewards = []
        for start in start_states:
            s = env.reset(pos=(start//5, start%5))
            total_r = 0.0
            for _ in range(H):
                s_ohe = np.eye(25)[s].reshape(1, -1)
                a = clf.predict(s_ohe)[0]
                s, r, done = env.step(a)
                total_r += r
                if done:
                    break
            rewards.append(total_r)
        results[H] = np.mean(rewards)
    return results


def evaluate_optimal_over_horizon(start_states, horizon_lengths):
    env = GridWorld()
    results = {}
    for H in horizon_lengths:
        rewards = []
        for start in start_states:
            s = env.reset(pos=(start//5, start%5))
            total_r = 0.0
            for _ in range(H):
                a = env.optimal_action(s)
                s, r, done = env.step(a)
                total_r += r
                if done:
                    break
            rewards.append(total_r)
        results[H] = np.mean(rewards)
    return results


start_states = list(range(0, 25, 5))  # diverse start positions
horizon_lengths = [5, 10, 15, 20, 30, 40, 50]

# Train BC with limited demos
np.random.seed(42)
few_demos = generate_expert_demos(n_trajectories=5, noise=0.05)
clf_few, _ = train_bc(few_demos)
many_demos = generate_expert_demos(n_trajectories=50, noise=0.05)
clf_many, _ = train_bc(many_demos)

bc_few_perf = evaluate_bc_over_horizon(clf_few, start_states, horizon_lengths)
bc_many_perf = evaluate_bc_over_horizon(clf_many, start_states, horizon_lengths)
opt_perf = evaluate_optimal_over_horizon(start_states, horizon_lengths)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

H_vals = horizon_lengths
axes[0].plot(H_vals, [opt_perf[h] for h in H_vals], 'o-',
             color='#1a9641', label='Optimal policy', linewidth=2)
axes[0].plot(H_vals, [bc_many_perf[h] for h in H_vals], 's-',
             color='#2c7bb6', label='BC (50 demos)', linewidth=2)
axes[0].plot(H_vals, [bc_few_perf[h] for h in H_vals], '^--',
             color='#d7191c', label='BC (5 demos)', linewidth=2)
axes[0].set_xlabel('Evaluation Horizon (steps)', fontsize=12)
axes[0].set_ylabel('Mean Reward', fontsize=12)
axes[0].set_title('BC Performance Degrades with Horizon Length', fontsize=12)
axes[0].legend()
axes[0].set_facecolor('#f8f8f8')
axes[0].grid(True, alpha=0.4)

# Show compounding error ratio
gap_few = [opt_perf[h] - bc_few_perf[h] for h in H_vals]
gap_many = [opt_perf[h] - bc_many_perf[h] for h in H_vals]
axes[1].plot(H_vals, gap_few, 'o-', color='#d7191c', label='BC few demos gap', linewidth=2)
axes[1].plot(H_vals, gap_many, 's-', color='#2c7bb6', label='BC many demos gap', linewidth=2)
axes[1].set_xlabel('Horizon Length', fontsize=12)
axes[1].set_ylabel('Optimal - BC Reward Gap', fontsize=12)
axes[1].set_title('Distributional Shift: Gap Widens with Horizon', fontsize=12)
axes[1].legend()
axes[1].set_facecolor('#f8f8f8')
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('/tmp/irl_bc_compounding.png', dpi=100, bbox_inches='tight')
plt.show()
print('Key: BC gap grows with horizon (distributional shift)')
print(f'BC few-demo gap at H=5: {gap_few[0]:.3f}, H=50: {gap_few[-1]:.3f}')


## Real-World Example 2: GAIL Approximation

GAIL uses a discriminator D(s,a) to distinguish expert vs agent trajectories.
The reward r = -log(1 - D(s,a)) guides the agent without explicit reward function.
This approximates IRL using adversarial training.

Note: Tabular GAIL on a small GridWorld requires many more iterations than neural GAIL
because Q-propagation is slow without function approximation. We demonstrate the
mechanism (discriminator + Q-learning) and show Q-value heatmap to diagnose convergence.
In practice, GAIL uses neural networks that generalize across states.


In [ ]:
def run_gail_approx(expert_demos, n_steps=50, alpha=0.1, gamma=0.9):
    env = GridWorld()
    # Q-table for agent policy
    Q = np.zeros((env.n_states, env.n_actions))

    # Discriminator: trained on (state, action) expert vs agent
    # Features: one-hot(state) + one-hot(action)
    def sa_features(s, a):
        feat = np.zeros(env.n_states + env.n_actions)
        feat[s] = 1.0
        feat[env.n_states + a] = 1.0
        return feat

    # Expert (s, a) pairs
    expert_sa = [(s, a) for traj in expert_demos for s, a in traj]

    performance_history = []
    discriminator = LogisticRegression(C=1.0, max_iter=200, random_state=42,
                                       warm_start=True)
    disc_trained = False  # initialize before loop

    for step in range(n_steps):
        # Collect agent trajectory AND update Q simultaneously (online Q-learning)
        agent_sa = []
        s = env.reset()
        done = False
        t = 0
        eps = max(0.05, 0.8 - step * 0.01)  # epsilon-greedy
        while not done and t < 50:
            if np.random.random() < eps:
                a = np.random.randint(env.n_actions)
            else:
                a = np.argmax(Q[s])
            agent_sa.append((s, a))
            s_next, _, done = env.step(a)

            # GAIL reward: -log(1 - D(s,a)); goal bonus to guide exploration
            if step > 5 and disc_trained:
                feat = sa_features(s, a).reshape(1, -1)
                try:
                    p_expert = discriminator.predict_proba(feat)[0, 1]
                    gail_r = -np.log(1 - p_expert + 1e-8)
                except Exception:
                    gail_r = 0.0
            else:
                gail_r = 0.0
            if done:
                gail_r += 2.0  # explicit goal bonus to guide Q-learning

            Q[s, a] += alpha * (gail_r + gamma * np.max(Q[s_next]) - Q[s, a])
            s = s_next
            t += 1

        # Train discriminator: expert=1, agent=0
        if len(agent_sa) > 5:
            n_each = min(len(expert_sa), len(agent_sa), 50)
            exp_idx = np.random.choice(len(expert_sa), n_each, replace=True)
            agt_idx = np.random.choice(len(agent_sa), n_each, replace=True)
            X_d = np.array([sa_features(expert_sa[i][0], expert_sa[i][1])
                            for i in exp_idx] +
                           [sa_features(agent_sa[i][0], agent_sa[i][1])
                            for i in agt_idx])
            y_d = np.array([1] * n_each + [0] * n_each)
            try:
                discriminator.fit(X_d, y_d)
                disc_trained = True
            except Exception:
                disc_trained = False

        # Evaluate success rate
        successes = 0
        env_eval = GridWorld()
        for _ in range(50):
            sv = env_eval.reset()
            dv = False
            for _ in range(30):
                av = np.argmax(Q[sv])
                sv, _, dv = env_eval.step(av)
                if dv:
                    successes += 1
                    break
        performance_history.append(successes / 50)

    return Q, performance_history


np.random.seed(42)
print('Running GAIL approximation (300 steps)...')
Q_gail, gail_perf = run_gail_approx(expert_demos, n_steps=300)
print(f'GAIL final success rate: {np.mean(gail_perf[-10:]):.2f}')
print(f'GAIL Q max: {Q_gail.max():.4f} (non-zero means Q-learning is working)')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(gail_perf, color='#fd8d3c', linewidth=2, label='GAIL approx')
# BC baseline
bc_sr, _ = evaluate_bc_policy(clf_many)
ax.axhline(y=bc_sr, color='#2c7bb6', linestyle='--', linewidth=2,
           label=f'BC (50 demos): {bc_sr:.2f}')
ax.set_xlabel('GAIL Training Step', fontsize=12)
ax.set_ylabel('Success Rate', fontsize=12)
ax.set_title('GAIL Learning Curve vs BC Baseline', fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_facecolor('#f8f8f8')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('/tmp/irl_gail.png', dpi=100, bbox_inches='tight')
plt.show()

# GAIL Q-value heatmap: shows which states have learned signal
# Q max per state indicates where Q-learning propagated
q_max_per_state = Q_gail.max(axis=1)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(q_max_per_state.reshape(5, 5), cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=ax)
ax.set_title('GAIL: Max Q per State\n(non-zero = Q-learning reached this state)', fontsize=11)
for i in range(5):
    for j in range(5):
        v = q_max_per_state[i * 5 + j]
        ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=8,
                color='white' if v > q_max_per_state.max() / 2 else 'black')
plt.tight_layout()
plt.savefig('/tmp/irl_gail_qheatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('Key insight: GAIL Q-learning needs more steps to propagate reward')
print('through all grid states. Neural GAIL with function approximation')
print('generalizes across states without visiting each one individually.')


## Real-World Example 3: Goal-Conditioned Reward Recovery

IRL for goal-reaching: expert demonstrations reach specific target cells.
Recover a reward function that generalizes to novel start/goal combinations.
This mirrors robotics-style IRL from goal-reaching demonstrations.


In [ ]:
def generate_goal_demos(goal_pos=(4,4), n_traj=15, noise=0.05):
    class GoalGridWorld(GridWorld):
        def __init__(self, goal):
            super().__init__()
            self.goal = goal

    env = GoalGridWorld(goal=goal_pos)
    demos = []
    for _ in range(n_traj):
        # Random start
        start_r = np.random.randint(env.rows - 1)  # not goal row
        start_c = np.random.randint(env.cols)
        s = env.reset(pos=(start_r, start_c))
        traj = []
        done = False
        for _ in range(40):
            if np.random.random() < noise:
                a = np.random.randint(env.n_actions)
            else:
                a = env.optimal_action(s)
            traj.append((s, a))
            s, _, done = env.step(a)
            if done:
                break
        demos.append(traj)
    return demos


# Recover reward for two different goals
np.random.seed(42)
demos_goal1 = generate_goal_demos(goal_pos=(4, 4), n_traj=15)
demos_goal2 = generate_goal_demos(goal_pos=(4, 0), n_traj=15)

print('MaxEnt IRL for goal (4,4)...')
reward_goal1, _ = run_maxent_irl(demos_goal1, n_iters=50)
print('MaxEnt IRL for goal (4,0)...')
reward_goal2, _ = run_maxent_irl(demos_goal2, n_iters=50)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, rw, goal, title in [
    (axes[0], reward_goal1, (4, 4), 'IRL: Expert goal = (4,4)'),
    (axes[1], reward_goal2, (4, 0), 'IRL: Expert goal = (4,0)'),
]:
    im = ax.imshow(rw.reshape(5, 5), cmap='RdYlGn', vmin=rw.min(), vmax=rw.max())
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax)
    # Mark the inferred goal (highest reward state)
    best_s = np.argmax(rw)
    ax.scatter(best_s % 5, best_s // 5, s=200, color='blue',
               marker='*', label=f'Max reward: ({best_s//5},{best_s%5})',
               zorder=5)
    ax.scatter(goal[1], goal[0], s=200, color='red',
               marker='x', linewidths=3, label=f'True goal: {goal}',
               zorder=5)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/irl_goal_conditioned.png', dpi=100, bbox_inches='tight')
plt.show()

goal1_best = np.argmax(reward_goal1)
goal2_best = np.argmax(reward_goal2)
print(f'Goal 1 (4,4): Max reward state = ({goal1_best//5},{goal1_best%5})')
print(f'Goal 2 (4,0): Max reward state = ({goal2_best//5},{goal2_best%5})')


## Comparison: BC vs MaxEnt IRL vs GAIL

Data efficiency and distribution shift robustness across all three methods.


In [ ]:
# Compare all methods on data efficiency (success rate vs num demos)
demo_counts_eval = [5, 10, 20, 30, 50]
bc_success_rates = []
irl_success_rates = []

for n_demos in demo_counts_eval:
    np.random.seed(0)
    demos_eval = generate_expert_demos(n_trajectories=n_demos, noise=0.05)

    # BC
    clf_eval, _ = train_bc(demos_eval)
    succ_bc, _ = evaluate_bc_policy(clf_eval)
    bc_success_rates.append(succ_bc)

    # MaxEnt IRL: recover reward, derive greedy policy
    rw, _ = run_maxent_irl(demos_eval, n_iters=40)
    feat = np.eye(25)
    Q_irl = greedy_value_iteration(rw, feat)
    env_irl = GridWorld()
    irl_succ = 0
    for _ in range(100):
        s = env_irl.reset()
        done = False
        for _ in range(50):
            a = np.argmax(Q_irl[s])
            s, _, done = env_irl.step(a)
            if done:
                irl_succ += 1
                break
    irl_success_rates.append(irl_succ / 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(demo_counts_eval, bc_success_rates, 'o-', color='#2c7bb6',
             linewidth=2, markersize=8, label='Behavioral Cloning')
axes[0].plot(demo_counts_eval, irl_success_rates, 's-', color='#d7191c',
             linewidth=2, markersize=8, label='MaxEnt IRL')
axes[0].plot(demo_counts_eval, [np.mean(gail_perf[-10:])] * len(demo_counts_eval),
             '--', color='#fd8d3c', linewidth=2, label=f'GAIL approx (fixed)')
axes[0].set_xlabel('Number of Expert Demonstrations', fontsize=12)
axes[0].set_ylabel('Success Rate', fontsize=12)
axes[0].set_title('Data Efficiency: BC vs IRL vs GAIL', fontsize=12)
axes[0].legend()
axes[0].set_ylim(0, 1.05)
axes[0].set_facecolor('#f8f8f8')
axes[0].grid(True, alpha=0.4)

# Summary bar: final performance at 20 demos
idx_20 = demo_counts_eval.index(20)
final_perf = [bc_success_rates[idx_20], irl_success_rates[idx_20], np.mean(gail_perf[-10:])]
method_names = ['Behavioral\nCloning', 'MaxEnt IRL', 'GAIL approx']
colors_final = ['#2c7bb6', '#d7191c', '#fd8d3c']
bars = axes[1].bar(method_names, final_perf, color=colors_final, alpha=0.85, edgecolor='black')
axes[1].set_ylabel('Success Rate (20 demos)', fontsize=12)
axes[1].set_title('Final Performance at 20 Demonstrations', fontsize=12)
axes[1].set_ylim(0, 1.1)
axes[1].set_facecolor('#f8f8f8')
axes[1].grid(True, alpha=0.4, axis='y')
for bar, val in zip(bars, final_perf):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/irl_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


## Key Takeaways

**Core idea:** IRL infers the reward function R that rationalizes expert behavior, then trains a new policy on the inferred R. BC skips reward inference; GAIL approximates IRL without solving the inner RL problem.

**Variants and when to use:**

| Method | RL inner loop | Distribution shift | Data efficiency | Use when |
|--------|--------------|-------------------|-----------------|----------|
| BC | No | Poor (O(T^2)) | High | Short horizons, abundant data |
| MaxEnt IRL | Yes | Good | Medium | Need transferable reward |
| GAIL | Yes (online) | Good | Low | High-dim, no explicit reward |
| DAgger | No (interactive) | Good | Medium | Online data collection possible |

**Common failure modes:**
- BC: T=10 works, T=100 fails (compounding O(T^2) errors)
- IRL: inner loop under-converged -> garbage reward weights
- GAIL: discriminator mode collapse -> D=1 everywhere, reward = 0


## Exercises

1. **BC horizon limit**: In `evaluate_bc_over_horizon`, at what horizon H does the BC policy first fail to beat random? What's the inflection point?
2. **IRL inner loop depth**: In `run_maxent_irl`, reduce `n_iter` in `softmax_value_iteration` to 5 (from 20). How does recovered reward quality change?
3. **Noisy expert BC**: Generate demos with noise=0.3 (30% random actions). Does BC performance degrade? Does IRL degrade as much?
4. **GAIL without RL**: Remove Q-learning from GAIL, keep only the discriminator reward. Show that GAIL needs RL to work; discrimination alone is insufficient.
